In [1]:
# ==============================================================================
# TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT
# NOTEBOOK: 09A_baseline_logloss_checkpoint.ipynb
# CELL 0 — ENVIRONMENT / PATHS / FROZEN LABEL + OOF CONFIG
# ==============================================================================

import gc
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd


print("=" * 90)
print("TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT")
print("CELL 0 — ENVIRONMENT / PATHS / FROZEN LABEL + OOF CONFIG")
print("=" * 90)


# ==============================================================================
# 1. PROJECT ROOT
# ==============================================================================

PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

assert PROJECT_ROOT.exists(), (
    "PROJECT_ROOT does not exist:\n"
    f"{PROJECT_ROOT}"
)


# ==============================================================================
# 2. CORE DIRECTORIES
# ==============================================================================

SCRATCH_ROOT = (
    PROJECT_ROOT
    / "scratch_mastery_outputs"
)

EVIDENCE_ROOT = (
    SCRATCH_ROOT
    / "03_evidence_pack"
)

EVIDENCE_FREEZE_ROOT = (
    EVIDENCE_ROOT
    / "frozen"
)

BASELINE_ROOT = (
    SCRATCH_ROOT
    / "09A_baseline_logloss"
)

BASELINE_AUDIT_ROOT = (
    BASELINE_ROOT
    / "audit"
)

BASELINE_OUTPUT_ROOT = (
    BASELINE_ROOT
    / "outputs"
)

BASELINE_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

BASELINE_OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ==============================================================================
# 3. FROZEN EVIDENCE / LABEL ARTIFACT
# ==============================================================================

FROZEN_EVIDENCE_PATH = (
    EVIDENCE_FREEZE_ROOT
    / "evidence_packs.parquet"
)

FROZEN_EVIDENCE_MANIFEST = (
    EVIDENCE_FREEZE_ROOT
    / "cell6_freeze_manifest.json"
)


assert FROZEN_EVIDENCE_PATH.exists(), (
    "Frozen evidence pack artifact is missing:\n"
    f"{FROZEN_EVIDENCE_PATH}"
)

assert FROZEN_EVIDENCE_MANIFEST.exists(), (
    "Frozen evidence pack manifest is missing:\n"
    f"{FROZEN_EVIDENCE_MANIFEST}"
)


# ==============================================================================
# 4. EXISTING BASELINE OOF DISCOVERY
# ==============================================================================

# We deliberately do NOT invent a baseline OOF path.
#
# First check whether an earlier notebook already exposed a path variable.
# Then check known project locations.
#
# If nothing is found, Cell 0 FAILS and we inspect the actual artifact rather
# than silently creating pseudo-OOF predictions.

OOF_PATH_CANDIDATES = []


for variable_name in [
    "BASELINE_OOF_PATH",
    "OOF_PREDICTIONS_PATH",
    "BASELINE_PREDICTIONS_PATH",
    "OOF_PATH",
]:

    if variable_name in globals():

        candidate = globals()[
            variable_name
        ]

        if candidate is not None:

            OOF_PATH_CANDIDATES.append(
                Path(candidate)
            )


# Known likely project locations.
OOF_PATH_CANDIDATES.extend(
    [

        # Generic baseline OOF locations
        (
            SCRATCH_ROOT
            / "baseline"
            / "oof_predictions.parquet"
        ),

        (
            SCRATCH_ROOT
            / "baseline"
            / "baseline_oof_predictions.parquet"
        ),

        (
            SCRATCH_ROOT
            / "01_baseline"
            / "oof_predictions.parquet"
        ),

        (
            SCRATCH_ROOT
            / "09A_baseline"
            / "oof_predictions.parquet"
        ),

        (
            SCRATCH_ROOT
            / "09A_baseline"
            / "outputs"
            / "baseline_oof_predictions.parquet"
        ),

        (
            SCRATCH_ROOT
            / "05_baseline"
            / "oof_predictions.parquet"
        ),

    ]
)


BASELINE_OOF_PATH = None

for candidate_path in OOF_PATH_CANDIDATES:

    if candidate_path.exists():

        BASELINE_OOF_PATH = (
            candidate_path
        )

        break


# ==============================================================================
# 5. AUTHORITATIVE FOLD MANIFEST DISCOVERY
# ==============================================================================

FOLD_MANIFEST_CANDIDATES = []

for variable_name in [
    "FOLD_MANIFEST_PATH",
    "SPLIT_MANIFEST_PATH",
    "FROZEN_FOLD_MANIFEST_PATH",
]:

    if variable_name in globals():

        candidate = globals()[
            variable_name
        ]

        if candidate is not None:

            FOLD_MANIFEST_CANDIDATES.append(
                Path(candidate)
            )


FOLD_MANIFEST_CANDIDATES.extend(
    [

        (
            SCRATCH_ROOT
            / "00_setup"
            / "frozen_fold_manifest.parquet"
        ),

        (
            SCRATCH_ROOT
            / "00_environment"
            / "frozen_fold_manifest.parquet"
        ),

        (
            SCRATCH_ROOT
            / "01_data_foundation"
            / "frozen_fold_manifest.parquet"
        ),

        (
            SCRATCH_ROOT
            / "fold_manifest.parquet"
        ),

    ]
)


FOLD_MANIFEST_PATH = None

for candidate_path in FOLD_MANIFEST_CANDIDATES:

    if candidate_path.exists():

        FOLD_MANIFEST_PATH = (
            candidate_path
        )

        break


# ==============================================================================
# 6. PRINT PATH CONFIGURATION
# ==============================================================================

print("\n" + "=" * 90)
print("PROJECT PATHS")
print("=" * 90)

print(
    "PROJECT_ROOT       :",
    PROJECT_ROOT,
)

print(
    "SCRATCH_ROOT       :",
    SCRATCH_ROOT,
)

print(
    "EVIDENCE_ROOT      :",
    EVIDENCE_ROOT,
)

print(
    "BASELINE_ROOT      :",
    BASELINE_ROOT,
)

print(
    "BASELINE_AUDIT     :",
    BASELINE_AUDIT_ROOT,
)

print(
    "BASELINE_OUTPUT    :",
    BASELINE_OUTPUT_ROOT,
)


print("\n" + "=" * 90)
print("FROZEN EVIDENCE INPUT")
print("=" * 90)

print(
    "Evidence packs:",
    FROZEN_EVIDENCE_PATH,
)

print(
    "Evidence manifest:",
    FROZEN_EVIDENCE_MANIFEST,
)

print(
    "Evidence parquet:",
    "PASS",
)

print(
    "Evidence manifest:",
    "PASS",
)


# ==============================================================================
# 7. BASELINE OOF STATUS
# ==============================================================================

print("\n" + "=" * 90)
print("EXISTING BASELINE OOF")
print("=" * 90)

if BASELINE_OOF_PATH is None:

    print(
        "Baseline OOF predictions : NOT FOUND"
    )

    print(
        "This is intentional — no OOF artifact will be fabricated."
    )

else:

    print(
        "Baseline OOF predictions:",
        BASELINE_OOF_PATH,
    )

    print(
        "Baseline OOF predictions : FOUND"
    )


# ==============================================================================
# 8. FOLD MANIFEST STATUS
# ==============================================================================

print("\n" + "=" * 90)
print("FROZEN FOLD MANIFEST")
print("=" * 90)

if FOLD_MANIFEST_PATH is None:

    print(
        "Fold manifest : NOT FOUND"
    )

else:

    print(
        "Fold manifest:",
        FOLD_MANIFEST_PATH,
    )

    print(
        "Fold manifest : FOUND"
    )


# ==============================================================================
# 9. GLOBAL DATA CONTRACT
# ==============================================================================

EXPECTED_RESPONSES = 35_072
EXPECTED_SESSIONS = 22_821
EXPECTED_OBJECTIVES = 398
EXPECTED_FOLDS = 5

EXPECTED_POSITIVE = 24_637
EXPECTED_NEGATIVE = 10_435

EXPECTED_POSITIVE_RATE = (
    EXPECTED_POSITIVE
    /
    EXPECTED_RESPONSES
)


print("\n" + "=" * 90)
print("GLOBAL DATA CONTRACT")
print("=" * 90)

print(
    "Responses       :",
    f"{EXPECTED_RESPONSES:,}",
)

print(
    "Sessions        :",
    f"{EXPECTED_SESSIONS:,}",
)

print(
    "Objectives      :",
    f"{EXPECTED_OBJECTIVES:,}",
)

print(
    "Folds           :",
    EXPECTED_FOLDS,
)

print(
    "Positive labels :",
    f"{EXPECTED_POSITIVE:,}",
)

print(
    "Negative labels :",
    f"{EXPECTED_NEGATIVE:,}",
)

print(
    "Positive rate   :",
    f"{EXPECTED_POSITIVE_RATE:.6f}",
)


# ==============================================================================
# 10. BASELINE REFERENCE CONTRACT
# ==============================================================================

REFERENCE_OVERALL_LOGLOSS = 0.555460

REFERENCE_POSITIVE_LOGLOSS = 0.327

REFERENCE_NEGATIVE_LOGLOSS = 1.095

REFERENCE_MIXED_SESSION_LOGLOSS = 0.770

REFERENCE_MIXED_SESSION_MEAN_LOGLOSS = 0.782

REFERENCE_SAME_SESSION_PAIR_ACCURACY = 0.679

REFERENCE_MEDIAN_PAIR_MARGIN = 0.0051

REFERENCE_PAIR_COLLAPSE_RATE = 0.7323

REFERENCE_SESSION_COLLAPSE_RATE = 0.7493

REFERENCE_FALSE_POSITIVE_RATE_05 = 0.8104

REFERENCE_HIGH_CONFIDENCE_FALSE_POSITIVE_RATE = 0.1326


print("\n" + "=" * 90)
print("BASELINE REFERENCE CONTRACT")
print("=" * 90)

print(
    "Overall OOF Log Loss:",
    REFERENCE_OVERALL_LOGLOSS,
)

print(
    "Positive-class LL:",
    REFERENCE_POSITIVE_LOGLOSS,
)

print(
    "Negative-class LL:",
    REFERENCE_NEGATIVE_LOGLOSS,
)

print(
    "Mixed-session LL:",
    REFERENCE_MIXED_SESSION_LOGLOSS,
)

print(
    "Mixed-session mean LL:",
    REFERENCE_MIXED_SESSION_MEAN_LOGLOSS,
)

print(
    "Same-session pair accuracy:",
    REFERENCE_SAME_SESSION_PAIR_ACCURACY,
)

print(
    "Median pair margin:",
    REFERENCE_MEDIAN_PAIR_MARGIN,
)

print(
    "Pair-collapse rate:",
    REFERENCE_PAIR_COLLAPSE_RATE,
)

print(
    "Session-collapse rate:",
    REFERENCE_SESSION_COLLAPSE_RATE,
)

print(
    "FP rate @ 0.5:",
    REFERENCE_FALSE_POSITIVE_RATE_05,
)

print(
    "High-confidence FP rate:",
    REFERENCE_HIGH_CONFIDENCE_FALSE_POSITIVE_RATE,
)


# ==============================================================================
# 11. OUTPUT CONTRACT
# ==============================================================================

OOF_AUDIT_PATH = (
    BASELINE_AUDIT_ROOT
    / "baseline_oof_recomputed_metrics.parquet"
)

FOLD_METRICS_PATH = (
    BASELINE_AUDIT_ROOT
    / "fold_logloss_metrics.parquet"
)

PAIR_METRICS_PATH = (
    BASELINE_AUDIT_ROOT
    / "same_session_pair_metrics.parquet"
)

MIXED_SESSION_METRICS_PATH = (
    BASELINE_AUDIT_ROOT
    / "mixed_session_metrics.parquet"
)

BASELINE_CHECKPOINT_PATH = (
    BASELINE_OUTPUT_ROOT
    / "baseline_logloss_checkpoint.json"
)


print("\n" + "=" * 90)
print("OUTPUT PATHS")
print("=" * 90)

print(
    "OOF audit:",
    OOF_AUDIT_PATH,
)

print(
    "Fold metrics:",
    FOLD_METRICS_PATH,
)

print(
    "Pair metrics:",
    PAIR_METRICS_PATH,
)

print(
    "Mixed-session metrics:",
    MIXED_SESSION_METRICS_PATH,
)

print(
    "Checkpoint:",
    BASELINE_CHECKPOINT_PATH,
)


# ==============================================================================
# 12. CELL 0 CONFIGURATION
# ==============================================================================

LOGLOSS_EPSILON = 1e-7

PAIR_MARGIN_THRESHOLD = 0.0

HIGH_CONFIDENCE_THRESHOLD = 0.8

FP_THRESHOLD = 0.5

MIXED_SESSION_MIN_LABELS = 2

RANDOM_SEED = 42


# ==============================================================================
# 13. READY FLAG
# ==============================================================================

BASELINE_LOGLOSS_CELL_0_READY = True


print("\n" + "=" * 90)
print("09A CELL 0 — BOOTSTRAP STATUS")
print("=" * 90)

print(
    "Frozen evidence input : PASS"
)

print(
    "Metric configuration   : PASS"
)

print(
    "Reference baseline     : PASS"
)

print(
    "Output configuration   : PASS"
)

print(
    "Bootstrap ready        : PASS"
)


# ==============================================================================
# 14. MEMORY CLEANUP
# ==============================================================================

gc.collect()

print(
    "Cell 0 memory cleanup: PASS"
)

TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT
CELL 0 — ENVIRONMENT / PATHS / FROZEN LABEL + OOF CONFIG

PROJECT PATHS
PROJECT_ROOT       : D:\Competition\Trace-the-race-local
SCRATCH_ROOT       : D:\Competition\Trace-the-race-local\scratch_mastery_outputs
EVIDENCE_ROOT      : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack
BASELINE_ROOT      : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09A_baseline_logloss
BASELINE_AUDIT     : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09A_baseline_logloss\audit
BASELINE_OUTPUT    : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09A_baseline_logloss\outputs

FROZEN EVIDENCE INPUT
Evidence packs: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\evidence_packs.parquet
Evidence manifest: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\03_evidence_pack\frozen\cell6_freeze_manifest.json
Evidence parquet: PASS
Evidence manifest: PASS

EXI

In [5]:
# ==============================================================================
# TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT
# CELL 1 — EXISTING OOF ARTIFACT DISCOVERY
# ==============================================================================

import gc
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT")
print("CELL 1 — EXISTING OOF ARTIFACT DISCOVERY")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY
# ==============================================================================

assert (
    "BASELINE_LOGLOSS_CELL_0_READY" in globals()
    and BASELINE_LOGLOSS_CELL_0_READY is True
), "Cell 0 dependency failed."

print("\nCell 0 dependency : PASS")


# ==============================================================================
# 2. SEARCH ROOT
# ==============================================================================

SEARCH_ROOT = Path(
    PROJECT_ROOT
)

assert SEARCH_ROOT.exists(), (
    "Project root does not exist."
)


# ==============================================================================
# 3. SEARCH TERMS
# ==============================================================================

SEARCH_TERMS = [
    "oof",
    "prediction",
    "predictions",
    "pred",
    "baseline",
    "logloss",
    "tfidf",
    "neural",
]


# ==============================================================================
# 4. SEARCH FILES
# ==============================================================================

print("\n" + "=" * 90)
print("PROJECT OOF / PREDICTION ARTIFACT SEARCH")
print("=" * 90)

matches = []

for path in SEARCH_ROOT.rglob("*"):

    if not path.is_file():
        continue

    path_lower = str(path).lower()

    # Ignore environments / caches / notebook checkpoints.
    ignored_tokens = [
        "\\.git\\",
        "\\__pycache__\\",
        "\\.venv\\",
        "\\venv\\",
        "\\node_modules\\",
        "\\.ipynb_checkpoints\\",
        "\\.cache\\",
    ]

    if any(
        token in path_lower
        for token in ignored_tokens
    ):
        continue

    suffix = path.suffix.lower()

    if suffix not in {
        ".parquet",
        ".csv",
        ".json",
    }:
        continue

    filename_lower = path.name.lower()

    if any(
        term in filename_lower
        for term in SEARCH_TERMS
    ):
        matches.append(
            path
        )


matches = sorted(
    set(matches),
    key=lambda p: str(p).lower(),
)


print(
    "Candidate files found:",
    len(matches),
)


# ==============================================================================
# 5. INSPECT PARQUET / CSV SCHEMAS
# ==============================================================================

rows = []

for path in matches:

    try:

        suffix = path.suffix.lower()

        if suffix == ".parquet":

            pf = pq.ParquetFile(
                path
            )

            columns = (
                pf.schema_arrow.names
            )

            row_count = (
                pf.metadata.num_rows
            )

        elif suffix == ".csv":

            columns = (
                pd.read_csv(
                    path,
                    nrows=0,
                )
                .columns
                .tolist()
            )

            row_count = None

        else:

            columns = []
            row_count = None


        lower_columns = {
            str(column).lower()
            for column in columns
        }

        has_response_id = (
            "response_id"
            in lower_columns
        )

        has_prediction = any(
            column in lower_columns
            for column in [
                "prediction",
                "pred",
                "probability",
                "prob",
                "oof_prediction",
                "oof_pred",
            ]
        )

        has_fold = (
            "fold"
            in lower_columns
            or
            "cv_fold"
            in lower_columns
            or
            "oof_fold"
            in lower_columns
        )

        has_target = any(
            column in lower_columns
            for column in [
                "target",
                "label",
                "y",
            ]
        )

        score = (
            int(has_response_id)
            +
            int(has_prediction)
            +
            int(has_fold)
            +
            int(has_target)
        )

        rows.append(
            {
                "path": str(path),
                "filename": path.name,
                "suffix": suffix,
                "rows": row_count,
                "columns": ", ".join(
                    map(
                        str,
                        columns,
                    )
                ),
                "has_response_id": has_response_id,
                "has_prediction": has_prediction,
                "has_fold": has_fold,
                "has_target": has_target,
                "schema_score": score,
                "size_mb": round(
                    path.stat().st_size
                    /
                    (1024 ** 2),
                    2,
                ),
            }
        )

    except Exception as exc:

        rows.append(
            {
                "path": str(path),
                "filename": path.name,
                "suffix": path.suffix.lower(),
                "rows": None,
                "columns": f"ERROR: {exc}",
                "has_response_id": False,
                "has_prediction": False,
                "has_fold": False,
                "has_target": False,
                "schema_score": -1,
                "size_mb": round(
                    path.stat().st_size
                    /
                    (1024 ** 2),
                    2,
                ),
            }
        )


discovery_df = pd.DataFrame(
    rows,
    columns=[
        "path",
        "filename",
        "suffix",
        "rows",
        "columns",
        "has_response_id",
        "has_prediction",
        "has_fold",
        "has_target",
        "schema_score",
        "size_mb",
    ],
)


# ==============================================================================
# 6. DISPLAY BEST CANDIDATES
# ==============================================================================

print("\n" + "=" * 90)
print("BEST OOF CANDIDATES")
print("=" * 90)

if discovery_df.empty:

    print(
        "NO candidate OOF/prediction artifacts found."
    )

else:

    best_candidates = (
        discovery_df
        .sort_values(
            [
                "schema_score",
                "rows",
                "size_mb",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )

    display(
        best_candidates.head(
            30
        )
    )


# ==============================================================================
# 7. STRONG OOF CANDIDATES
# ==============================================================================

strong_candidates = (
    discovery_df[
        (
            discovery_df[
                "has_response_id"
            ]
        )
        &
        (
            discovery_df[
                "has_prediction"
            ]
        )
    ]
    .sort_values(
        [
            "schema_score",
            "rows",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print("\n" + "=" * 90)
print("STRONG OOF CANDIDATES")
print("=" * 90)

print(
    "Candidates with response_id + prediction:",
    len(strong_candidates),
)

if not strong_candidates.empty:

    display(
        strong_candidates
    )

else:

    print(
        "No strong OOF candidate found."
    )


# ==============================================================================
# 8. SAVE DISCOVERY AUDIT
# ==============================================================================

DISCOVERY_AUDIT_PATH = (
    BASELINE_AUDIT_ROOT
    / "oof_artifact_discovery.parquet"
)

discovery_df.to_parquet(
    DISCOVERY_AUDIT_PATH,
    index=False,
    compression="zstd",
)


# ==============================================================================
# 9. DO NOT AUTO-SELECT
# ==============================================================================

if len(strong_candidates) == 1:

    print("\n" + "=" * 90)
    print("DISCOVERY STATUS")
    print("=" * 90)

    print(
        "Exactly one strong candidate found:"
    )

    print(
        strong_candidates.loc[
            0,
            "path",
        ]
    )

    print(
        "\nThis candidate is NOT automatically accepted."
    )

    print(
        "It must pass exact identity/schema validation in Cell 2."
    )

else:

    print("\n" + "=" * 90)
    print("DISCOVERY STATUS")
    print("=" * 90)

    print(
        "Automatic OOF selection : BLOCKED"
    )

    print(
        "Reason:",
        (
            "zero candidates"
            if len(strong_candidates) == 0
            else
            f"{len(strong_candidates)} candidates require disambiguation"
        )
    )


# ==============================================================================
# 10. READY FLAG
# ==============================================================================

BASELINE_LOGLOSS_CELL_1_READY = True

print(
    "\n09A CELL 1 — OOF ARTIFACT DISCOVERY: PASS"
)

print(
    "Discovery audit:",
    DISCOVERY_AUDIT_PATH,
)

print(
    "Cell 1 memory cleanup: PASS"
)

gc.collect()

TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT
CELL 1 — EXISTING OOF ARTIFACT DISCOVERY

Cell 0 dependency : PASS

PROJECT OOF / PREDICTION ARTIFACT SEARCH
Candidate files found: 10

BEST OOF CANDIDATES


,path,filename,suffix,rows,columns,has_response_id,has_prediction,has_fold,has_target,schema_score,size_mb
0,D:\Competition\Trace-the-race-local\Dataset\tr...,cguhoof.csv,.csv,None,"session_id, utterance_id, role, content, times...",False,False,False,False,0,0.03
1,D:\Competition\Trace-the-race-local\Dataset\tr...,gooofir.csv,.csv,None,"session_id, utterance_id, role, content, times...",False,False,False,False,0,0.03
2,D:\Competition\Trace-the-race-local\Dataset\tr...,hbrjoof.csv,.csv,None,"session_id, utterance_id, role, content, times...",False,False,False,False,0,0.03
3,D:\Competition\Trace-the-race-local\Dataset\tr...,imwoofw.csv,.csv,None,"session_id, utterance_id, role, content, times...",False,False,False,False,0,0.03
4,D:\Competition\Trace-the-race-local\scratch_ma...,tfidf_config.json,.json,None,,False,False,False,False,0,0.00
5,D:\Competition\Trace-the-race-local\scratch_ma...,tfidf_config.json,.json,None,,False,False,False,False,0,0.00
6,D:\Competition\Trace-the-race-local\scratch_ma...,tfidf_config.json,.json,None,,False,False,False,False,0,0.00
7,D:\Competition\Trace-the-race-local\scratch_ma...,tfidf_config.json,.json,None,,False,False,False,False,0,0.00
8,D:\Competition\Trace-the-race-local\scratch_ma...,tfidf_config.json,.json,None,,False,False,False,False,0,0.00
9,D:\Competition\Trace-the-race-local\scratch_ma...,tfidf_manifest.json,.json,None,,False,False,False,False,0,0.00



STRONG OOF CANDIDATES
Candidates with response_id + prediction: 0
No strong OOF candidate found.

DISCOVERY STATUS
Automatic OOF selection : BLOCKED
Reason: zero candidates

09A CELL 1 — OOF ARTIFACT DISCOVERY: PASS
Discovery audit: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\09A_baseline_logloss\audit\oof_artifact_discovery.parquet
Cell 1 memory cleanup: PASS


0

In [6]:
# ==============================================================================
# TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT
# 09A_baseline_logloss_checkpoint.ipynb
#
# CELL 2 — RECORDED BASELINE METRIC CHECKPOINT + FROZEN LABEL CONTRACT
# ==============================================================================

import gc
import json
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


print("=" * 90)
print("TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT")
print("CELL 2 — RECORDED BASELINE METRIC CHECKPOINT + FROZEN LABEL CONTRACT")
print("=" * 90)


# ==============================================================================
# 1. DEPENDENCY GATE
# ==============================================================================

assert (
    "BASELINE_LOGLOSS_CELL_1_READY" in globals()
    and BASELINE_LOGLOSS_CELL_1_READY is True
), (
    "Cell 1 dependency failed."
)

print("\n" + "=" * 90)
print("DEPENDENCY GATE")
print("=" * 90)

print(
    "Cell 0 dependency : PASS"
)

print(
    "Cell 1 dependency : PASS"
)


# ==============================================================================
# 2. IMPORTANT METHODOLOGICAL CONTRACT
#
# No historical OOF prediction artifact was discovered in Cell 1.
#
# Therefore:
#
#   - We DO NOT recompute historical OOF Log Loss here.
#   - We DO NOT fabricate predictions.
#   - We DO NOT treat the recorded 0.55546 as a newly recomputed metric.
#
# All historical values below are explicitly recorded references.
# ==============================================================================

BASELINE_METRIC_STATUS = (
    "RECORDED_REFERENCE_NOT_RECOMPUTED"
)

OOF_PREDICTION_ARTIFACT_STATUS = (
    "NOT_AVAILABLE"
)


print("\n" + "=" * 90)
print("BASELINE METRIC PROVENANCE")
print("=" * 90)

print(
    "OOF prediction artifact:",
    OOF_PREDICTION_ARTIFACT_STATUS,
)

print(
    "Historical metric status:",
    BASELINE_METRIC_STATUS,
)

assert (
    OOF_PREDICTION_ARTIFACT_STATUS
    ==
    "NOT_AVAILABLE"
), (
    "Unexpected OOF artifact status."
)

assert (
    BASELINE_METRIC_STATUS
    ==
    "RECORDED_REFERENCE_NOT_RECOMPUTED"
), (
    "Unexpected baseline metric provenance."
)

print(
    "Metric provenance contract : PASS"
)


# ==============================================================================
# 3. FROZEN EVIDENCE INPUT
# ==============================================================================

assert FROZEN_EVIDENCE_PATH.exists(), (
    "Frozen evidence pack is missing:\n"
    f"{FROZEN_EVIDENCE_PATH}"
)

assert FROZEN_EVIDENCE_MANIFEST.exists(), (
    "Frozen evidence manifest is missing:\n"
    f"{FROZEN_EVIDENCE_MANIFEST}"
)


with open(
    FROZEN_EVIDENCE_MANIFEST,
    "r",
    encoding="utf-8",
) as handle:

    evidence_manifest = json.load(
        handle
    )


assert (
    evidence_manifest.get("status")
    ==
    "FROZEN"
), (
    "Evidence manifest is not FROZEN."
)

assert (
    evidence_manifest.get("artifact")
    ==
    "evidence_packs"
), (
    "Unexpected frozen evidence artifact."
)


# ==============================================================================
# 4. FROZEN EVIDENCE METADATA
# ==============================================================================

evidence_pf = pq.ParquetFile(
    FROZEN_EVIDENCE_PATH
)

evidence_rows = (
    evidence_pf.metadata.num_rows
)

evidence_columns = (
    evidence_pf.schema_arrow.names
)


assert (
    evidence_rows
    ==
    EXPECTED_RESPONSES
), (
    "Frozen evidence population mismatch.\n"
    f"Expected: {EXPECTED_RESPONSES:,}\n"
    f"Observed: {evidence_rows:,}"
)


REQUIRED_LABEL_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
]

missing_label_columns = sorted(
    set(REQUIRED_LABEL_COLUMNS)
    -
    set(evidence_columns)
)

assert not missing_label_columns, (
    "Frozen evidence is missing label columns:\n"
    f"{missing_label_columns}"
)


print("\n" + "=" * 90)
print("FROZEN EVIDENCE CONTRACT")
print("=" * 90)

print(
    "Evidence rows:",
    f"{evidence_rows:,}",
)

print(
    "Required label columns:",
    REQUIRED_LABEL_COLUMNS,
)

print(
    "Frozen evidence : PASS"
)


# ==============================================================================
# 5. LOAD ONLY LABEL / GROUP METADATA
#
# Do not load evidence_text.
# Do not load the 2048-token evidence payload.
# ==============================================================================

LABEL_COLUMNS = [
    "response_id",
    "session_id",
    "objective_uid",
    "fold",
    "target",
]

labels_df = pd.read_parquet(
    FROZEN_EVIDENCE_PATH,
    columns=LABEL_COLUMNS,
)


# ==============================================================================
# 6. NORMALIZE TYPES
# ==============================================================================

for column in [
    "response_id",
    "session_id",
    "objective_uid",
]:

    labels_df[column] = (
        labels_df[column]
        .astype(str)
        .str.strip()
    )


labels_df["fold"] = (
    pd.to_numeric(
        labels_df["fold"],
        errors="coerce",
    )
)

labels_df["target"] = (
    pd.to_numeric(
        labels_df["target"],
        errors="coerce",
    )
)


# ==============================================================================
# 7. LABEL POPULATION CONTRACT
# ==============================================================================

assert (
    len(labels_df)
    ==
    EXPECTED_RESPONSES
), (
    "Label population mismatch."
)

assert (
    labels_df["response_id"]
    .notna()
    .all()
), (
    "response_id contains null values."
)

assert (
    labels_df["session_id"]
    .notna()
    .all()
), (
    "session_id contains null values."
)

assert (
    labels_df["objective_uid"]
    .notna()
    .all()
), (
    "objective_uid contains null values."
)

assert (
    labels_df["fold"]
    .notna()
    .all()
), (
    "fold contains null values."
)

assert (
    labels_df["target"]
    .notna()
    .all()
), (
    "target contains null values."
)

assert (
    labels_df["response_id"]
    .is_unique
), (
    "response_id is not unique."
)

assert (
    set(
        labels_df["target"]
        .astype(int)
        .unique()
    )
    <=
    {0, 1}
), (
    "target contains values outside {0,1}."
)


# ==============================================================================
# 8. LABEL DISTRIBUTION
# ==============================================================================

labels_df["fold"] = (
    labels_df["fold"]
    .astype(int)
)

labels_df["target"] = (
    labels_df["target"]
    .astype(int)
)


target_counts = (
    labels_df["target"]
    .value_counts()
    .sort_index()
)

fold_counts = (
    labels_df["fold"]
    .value_counts()
    .sort_index()
)


assert (
    int(
        target_counts.get(0, 0)
    )
    ==
    EXPECTED_NEGATIVE
), (
    "Negative target count mismatch."
)

assert (
    int(
        target_counts.get(1, 0)
    )
    ==
    EXPECTED_POSITIVE
), (
    "Positive target count mismatch."
)

assert (
    set(
        fold_counts.index
    )
    ==
    {0, 1, 2, 3, 4}
), (
    "Frozen labels do not contain folds 0-4."
)


print("\n" + "=" * 90)
print("FROZEN LABEL DISTRIBUTION")
print("=" * 90)

print(
    "Rows:",
    f"{len(labels_df):,}",
)

print(
    "Target counts:",
    {
        int(k): int(v)
        for k, v in target_counts.items()
    },
)

print(
    "Fold counts:",
    {
        int(k): int(v)
        for k, v in fold_counts.items()
    },
)

print(
    "Frozen label contract : PASS"
)


# ==============================================================================
# 9. GROUPING CONTRACT
#
# The historical baseline is session-grouped.
# Verify that every response belongs to exactly one session and fold.
# ==============================================================================

response_session_counts = (
    labels_df
    .groupby(
        "response_id",
        sort=False,
    )["session_id"]
    .nunique()
)

assert (
    response_session_counts
    .max()
    ==
    1
), (
    "A response_id maps to multiple sessions."
)


response_fold_counts = (
    labels_df
    .groupby(
        "response_id",
        sort=False,
    )["fold"]
    .nunique()
)

assert (
    response_fold_counts
    .max()
    ==
    1
), (
    "A response_id maps to multiple folds."
)


session_fold_counts = (
    labels_df
    .groupby(
        "session_id",
        sort=False,
    )["fold"]
    .nunique()
)

assert (
    session_fold_counts
    .max()
    ==
    1
), (
    "A session appears in multiple folds."
)


print("\n" + "=" * 90)
print("SESSION-GROUPED FOLD CONTRACT")
print("=" * 90)

print(
    "Response → single session : PASS"
)

print(
    "Response → single fold    : PASS"
)

print(
    "Session → single fold     : PASS"
)

print(
    "Session-grouped contract  : PASS"
)


# ==============================================================================
# 10. RECORDED HISTORICAL BASELINE
#
# These are locked reference values, NOT recomputed in this notebook.
# ==============================================================================

RECORDED_BASELINE = {
    "overall_oof_logloss": 0.555460,
    "positive_class_logloss": 0.327,
    "negative_class_logloss": 1.095,
    "mixed_session_response_logloss": 0.770,
    "mixed_session_session_mean_logloss": 0.782,
    "same_session_pair_accuracy": 0.679,
    "median_same_session_pair_margin": 0.0051,
    "pair_collapse_rate": 0.7323,
    "session_collapse_rate": 0.7493,
    "false_positive_rate_at_0_5": 0.8104,
    "high_confidence_false_positive_rate": 0.1326,
}


# ==============================================================================
# 11. RECORDED BASELINE SANITY CHECK
# ==============================================================================

assert (
    0.0
    <
    RECORDED_BASELINE[
        "overall_oof_logloss"
    ]
), (
    "Recorded Log Loss must be positive."
)

assert (
    0.0
    <=
    RECORDED_BASELINE[
        "same_session_pair_accuracy"
    ]
    <=
    1.0
), (
    "Recorded pair accuracy outside [0,1]."
)

assert (
    0.0
    <=
    RECORDED_BASELINE[
        "pair_collapse_rate"
    ]
    <=
    1.0
), (
    "Recorded pair collapse rate outside [0,1]."
)

assert (
    0.0
    <=
    RECORDED_BASELINE[
        "session_collapse_rate"
    ]
    <=
    1.0
), (
    "Recorded session collapse rate outside [0,1]."
)

assert (
    0.0
    <=
    RECORDED_BASELINE[
        "false_positive_rate_at_0_5"
    ]
    <=
    1.0
), (
    "Recorded FP rate outside [0,1]."
)

assert (
    0.0
    <=
    RECORDED_BASELINE[
        "high_confidence_false_positive_rate"
    ]
    <=
    1.0
), (
    "Recorded high-confidence FP rate outside [0,1]."
)


# ==============================================================================
# 12. DISPLAY RECORDED BASELINE
# ==============================================================================

print("\n" + "=" * 90)
print("RECORDED HISTORICAL BASELINE")
print("=" * 90)

print(
    "Overall OOF Log Loss              :",
    RECORDED_BASELINE[
        "overall_oof_logloss"
    ],
)

print(
    "Positive-class Log Loss           :",
    RECORDED_BASELINE[
        "positive_class_logloss"
    ],
)

print(
    "Negative-class Log Loss           :",
    RECORDED_BASELINE[
        "negative_class_logloss"
    ],
)

print(
    "Mixed-session response Log Loss   :",
    RECORDED_BASELINE[
        "mixed_session_response_logloss"
    ],
)

print(
    "Mixed-session session-mean LL     :",
    RECORDED_BASELINE[
        "mixed_session_session_mean_logloss"
    ],
)

print(
    "Same-session pair accuracy        :",
    RECORDED_BASELINE[
        "same_session_pair_accuracy"
    ],
)

print(
    "Median pair margin                :",
    RECORDED_BASELINE[
        "median_same_session_pair_margin"
    ],
)

print(
    "Pair-collapse rate                :",
    RECORDED_BASELINE[
        "pair_collapse_rate"
    ],
)

print(
    "Session-collapse rate             :",
    RECORDED_BASELINE[
        "session_collapse_rate"
    ],
)

print(
    "False-positive rate @ 0.5         :",
    RECORDED_BASELINE[
        "false_positive_rate_at_0_5"
    ],
)

print(
    "High-confidence FP rate           :",
    RECORDED_BASELINE[
        "high_confidence_false_positive_rate"
    ],
)


print(
    "\nStatus: RECORDED REFERENCE — NOT RECOMPUTED"
)


# ==============================================================================
# 13. BASELINE CHECKPOINT PAYLOAD
# ==============================================================================

checkpoint_payload = {
    "artifact": "09A_baseline_logloss_checkpoint",

    "status": "CHECKPOINTED",

    "metric_provenance": (
        "RECORDED_REFERENCE_NOT_RECOMPUTED"
    ),

    "oof_prediction_artifact": {
        "status": "NOT_AVAILABLE",
        "recomputed": False,
    },

    "frozen_evidence": {
        "path": str(
            FROZEN_EVIDENCE_PATH
        ),
        "rows": int(
            len(labels_df)
        ),
        "positive_count": int(
            target_counts.get(1, 0)
        ),
        "negative_count": int(
            target_counts.get(0, 0)
        ),
        "fold_counts": {
            str(k): int(v)
            for k, v in fold_counts.items()
        },
    },

    "recorded_baseline": RECORDED_BASELINE,

    "primary_metric": {
        "name": "grouped_oof_logloss",
        "direction": "lower_is_better",
        "recorded_value": (
            RECORDED_BASELINE[
                "overall_oof_logloss"
            ]
        ),
    },

    "evaluation_contract": {
        "folds": 5,
        "group_column": "session_id",
        "primary_metric": "log_loss",
        "same_session_pair_metrics": True,
        "mixed_session_metrics": True,
    },

    "next_actual_recomputation": (
        "ModernBERT OOF predictions or another "
        "explicitly generated baseline OOF artifact "
        "must be scored from predictions."
    ),

    "created_at_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}


# ==============================================================================
# 14. CHECKPOINT JSON
# ==============================================================================

assert BASELINE_CHECKPOINT_PATH.parent.exists(), (
    "Baseline output directory does not exist."
)

with open(
    BASELINE_CHECKPOINT_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        checkpoint_payload,
        handle,
        indent=2,
        sort_keys=True,
    )


assert (
    BASELINE_CHECKPOINT_PATH.exists()
), (
    "Baseline checkpoint was not created."
)


# ==============================================================================
# 15. CHECKPOINT RELOAD
# ==============================================================================

with open(
    BASELINE_CHECKPOINT_PATH,
    "r",
    encoding="utf-8",
) as handle:

    checkpoint_reload = json.load(
        handle
    )


assert (
    checkpoint_reload[
        "status"
    ]
    ==
    "CHECKPOINTED"
), (
    "Checkpoint status mismatch."
)

assert (
    checkpoint_reload[
        "metric_provenance"
    ]
    ==
    "RECORDED_REFERENCE_NOT_RECOMPUTED"
), (
    "Checkpoint provenance mismatch."
)

assert (
    checkpoint_reload[
        "recorded_baseline"
    ]
    ==
    RECORDED_BASELINE
), (
    "Checkpoint baseline values changed during serialization."
)


print("\n" + "=" * 90)
print("CHECKPOINT ARTIFACT")
print("=" * 90)

print(
    "Checkpoint:",
    BASELINE_CHECKPOINT_PATH,
)

print(
    "Checkpoint reload : PASS"
)


# ==============================================================================
# 16. FINAL CELL 2 CONTRACT
# ==============================================================================

BASELINE_LOGLOSS_CELL_2_READY = True

print("\n" + "=" * 90)
print(
    "09A CELL 2 — "
    "RECORDED BASELINE CHECKPOINT: PASS"
)
print("=" * 90)

print(
    "Frozen labels       : PASS"
)

print(
    "Session grouping    : PASS"
)

print(
    "Fold contract       : PASS"
)

print(
    "Historical baseline : LOCKED"
)

print(
    "OOF recomputation   : NOT PERFORMED"
)

print(
    "Checkpoint artifact : PASS"
)


# ==============================================================================
# 17. MEMORY CLEANUP
# ==============================================================================

for _name in [
    "evidence_pf",
    "evidence_manifest",
    "labels_df",
    "target_counts",
    "fold_counts",
    "response_session_counts",
    "response_fold_counts",
    "session_fold_counts",
    "checkpoint_reload",
]:
    if _name in globals():
        del globals()[_name]

gc.collect()

print(
    "Cell 2 memory cleanup: PASS"
)

TRACE THE ACE — BASELINE LOG-LOSS CHECKPOINT
CELL 2 — RECORDED BASELINE METRIC CHECKPOINT + FROZEN LABEL CONTRACT

DEPENDENCY GATE
Cell 0 dependency : PASS
Cell 1 dependency : PASS

BASELINE METRIC PROVENANCE
OOF prediction artifact: NOT_AVAILABLE
Historical metric status: RECORDED_REFERENCE_NOT_RECOMPUTED
Metric provenance contract : PASS

FROZEN EVIDENCE CONTRACT
Evidence rows: 35,072
Required label columns: ['response_id', 'session_id', 'objective_uid', 'fold', 'target']
Frozen evidence : PASS

FROZEN LABEL DISTRIBUTION
Rows: 35,072
Target counts: {0: 10435, 1: 24637}
Fold counts: {0: 6958, 1: 7050, 2: 7023, 3: 7081, 4: 6960}
Frozen label contract : PASS

SESSION-GROUPED FOLD CONTRACT
Response → single session : PASS
Response → single fold    : PASS
Session → single fold     : PASS
Session-grouped contract  : PASS

RECORDED HISTORICAL BASELINE
Overall OOF Log Loss              : 0.55546
Positive-class Log Loss           : 0.327
Negative-class Log Loss           : 1.095
Mixed-session